# LLM-Based Cluster Validation — DSPy Walkthrough

This notebook walks through evaluating and optimizing a DSPy program that solves an **intruder detection** task:
given six keywords where five belong to one semantic cluster and one is an outsider, the LLM must identify the intruder.

We go through four steps:
1. **Evaluate** — measure the baseline accuracy of the unoptimized program
2. **BootstrapFewShot** — add few-shot examples automatically from successful traces
3. **GEPA** — iteratively rewrite the prompt instructions using a reflection LM
4. **BootstrapFinetune** — distill a strong teacher (Claude Sonnet) into the student (ministral-3:3b) by updating weights

Each step logs to MLflow. Run `uv run mlflow ui` to explore results at http://localhost:5000.

## Prerequisites

Before running this notebook:
- Start the SGLang inference server serving `ministral-3:3b` on port 30000
- Set `ANTHROPIC_API_KEY` in your environment (needed for Sections 3 and 4)
- Run `python pipeline/build_dataset.py` at least once to generate `data/raw_examples.json`

```bash
python -m sglang.launch_server \
    --model-path ministral/Ministral-4b-instruct \
    --port 30000
```

## Setup

### What is DSPy?

DSPy is a framework for programming — rather than prompting — LLMs. Instead of hand-writing prompts,
you define a **Signature** (the input/output contract) and a **Module** (the reasoning strategy), and
DSPy handles prompt construction. Optimizers then automatically improve your program by searching over
few-shot examples, instructions, or even model weights.

Key abstractions used in this project:
- `dspy.Signature` — declares what goes in and what comes out (like a typed function signature)
- `dspy.ChainOfThought` — a predictor that asks the LM to reason step-by-step before answering
- `dspy.Evaluate` — runs the program over a dataset and computes a metric
- `dspy.Example` — a single labeled data point

In [1]:
import sys
sys.path.insert(0, ".")

import dspy
import mlflow

from cluster_validator import (
    ClusterIntruderValidator,
    IntruderDetectionSignature,
    build_devset,
    configure_dspy,
    configure_teacher_lm,
    intruder_exact_match,
    split_test,
    split_for_bootstrap,
    split_for_gepa,
    split_for_finetune,
)

CONFIG_PATH = "config/dspy_config.yaml"

# Configure the global DSPy LM (ministral-3:3b via SGLang)
configure_dspy(CONFIG_PATH, cache=True)

[dspy] Configured LM: lm_studio/Ministral-3b-instruct (cache=True)


### The program

The `IntruderDetectionSignature` below declares the task contract:
six keyword inputs and two outputs — step-by-step reasoning plus the intruder word.

`ClusterIntruderValidator` wraps that signature in a `dspy.ChainOfThought` predictor,
which instructs the LM to reason before answering. No hand-written prompt — DSPy constructs
the prompt automatically from the field names, descriptions, and any few-shot examples.

In [2]:
# Inspect the signature
print(IntruderDetectionSignature.__doc__)
print()

# Quick sanity check — run the unoptimized program on one example
program = ClusterIntruderValidator()
pred = program(
    trefwoord_1="ocean", trefwoord_2="river", trefwoord_3="lake",
    trefwoord_4="cat", trefwoord_5="stream", trefwoord_6="pond",
)
print(f"Reasoning : {pred.redenering}")
print(f"Intruder  : {pred.indringer}  (correct: mountain)")

Identify the one keyword that does not belong with the other five.

    You are given exactly six keywords. Five of them share a common semantic
    concept; one is an intruder that is conceptually different. Work only with the six
    provided words — do not introduce synonyms or related words that are not
    in the input. In two or three short sentences, state which concept the five
    words share and why the remaining word breaks that concept. Then output that
    word as the intruder.
    



2026/04/16 12:17:15 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.


Reasoning : The one word that does not belong with the others.
Intruder  : noun  (correct: mountain)


### The dataset

In [3]:
all_examples = build_devset()
print(f"Total examples: {len(all_examples)}")
print()

# Show the first example
ex = all_examples[0]
print("Example fields:", ex.inputs().keys())
print("Keywords :", [ex.trefwoord_1, ex.trefwoord_2, ex.trefwoord_3,
                     ex.trefwoord_4, ex.trefwoord_5, ex.trefwoord_6])
print("Intruder :", ex.indringer)

Loading examples from /home/rob/llm-based-cluster-validation/data/raw_examples.json …
  88 examples loaded.
Total examples: 88

Example fields: ['keyword_1', 'keyword_2', 'keyword_3', 'keyword_4', 'keyword_5', 'keyword_6']
Keywords : ['ek', 'finale', 'koeman', 'oranje', 'club', 'fiets']
Intruder : fiets


---

## Section 1: Evaluate the baseline

### How `dspy.Evaluate` works

`dspy.Evaluate` runs the program over a dataset in parallel, applies a metric function to each
(example, prediction) pair, and returns an aggregate score. Here the metric is `intruder_exact_match`:
it returns 1.0 when `prediction.intruder.strip().lower() == example.intruder.lower()`, else 0.0.

MLflow autologging captures every LM call as a trace, so you can inspect the full
prompt/response for any example in the MLflow UI.

In [4]:
from pipeline.evaluate import run_evaluation

baseline_score = run_evaluation(config_path=CONFIG_PATH, num_threads=4)
print(f"\nBaseline accuracy: {baseline_score:.1f}%")

[dspy] Configured LM: lm_studio/Ministral-3b-instruct (cache=True)


2026/04/16 12:18:14 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/16 12:18:14 INFO mlflow.store.db.utils: Updating database tables
2026/04/16 12:18:15 INFO mlflow.tracking.fluent: Experiment with name 'cluster-validator-eval' does not exist. Creating a new experiment.


Loading examples from /home/rob/llm-based-cluster-validation/data/raw_examples.json …
  88 examples loaded.
  0%|          | 0/88 [00:00<?, ?it/s]

2026/04/16 12:18:30 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:18:30 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:18:30 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.L

Average Metric: 0.00 / 4 (0.0%):   5%|▍         | 4/88 [00:15<03:23,  2.43s/it]

...


Accuracy: 1.1%  (1/88 correct)
Results saved to /home/rob/llm-based-cluster-validation/outputs/eval_results.json

--- Failure analysis (first 5 wrong) ---


2026/04/16 12:24:07 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:24:08 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.


  keywords : ['ek', 'finale', 'koeman', 'oranje', 'club', 'fiets']
  gold     : fiets
  pred     : intruder

  keywords : ['ek', 'verkiezingen', 'finale', 'koeman', 'oranje', 'club']
  gold     : verkiezingen
  pred     : intruder



2026/04/16 12:24:08 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:24:08 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.


  keywords : ['oranje', 'ek', 'koeman', 'club', 'a12', 'finale']
  gold     : a12
  pred     : an orange

  keywords : ['koeman', 'link to', 'ek', 'club', 'oranje', 'finale']
  gold     : link to
  pred     : intruder

  keywords : ['parlement', 'verkiezingen', 'europees parlement', 'stemmen', 'app', 'eu']
  gold     : app
  pred     : invincible


Baseline accuracy: 1.1%


---

## Section 2: BootstrapFewShot

### How BootstrapFewShot works

BootstrapFewShot improves a DSPy program **without touching model weights**.
It works in two steps:

1. **Bootstrap** — run the program on the training set, keep only the examples where
   the program answered correctly. These become candidate few-shot demonstrations.
2. **Select** — choose the best subset of demonstrations to prepend to the prompt
   (controlled by `max_bootstrapped_demos` and `max_labeled_demos`).

The result is a compiled program whose prompt now includes concrete worked examples.
No gradient descent — just prompt engineering, automated.

**Why a 20/80 train/dev split?**  
Prompt optimizers can overfit to a large training set (they memorise demonstrations rather
than generalising). DSPy recommends keeping training small and validation large so the
selected demonstrations transfer to unseen examples.

In [5]:
from pipeline.optimize import run_optimization as run_bootstrap

bootstrap_score = run_bootstrap(
    config_path=CONFIG_PATH,
    max_bootstrapped_demos=4,
    max_labeled_demos=16,
    num_threads=4,
)
print(f"\nBootstrapFewShot test accuracy: {bootstrap_score:.1f}%")
print(f"Delta vs baseline            : {bootstrap_score - baseline_score:+.1f}%")

2026/04/16 12:35:26 INFO mlflow.tracking.fluent: Experiment with name 'cluster-validator-optimize' does not exist. Creating a new experiment.


[dspy] Configured LM: lm_studio/Ministral-3b-instruct (cache=True)
Loading examples from /home/rob/llm-based-cluster-validation/data/raw_examples.json …
  88 examples loaded.
Split: 14 train / 60 dev / 14 test
  0%|          | 0/60 [00:00<?, ?it/s]

2026/04/16 12:35:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:35:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.
2026/04/16 12:35:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.L

Average Metric: 0.00 / 1 (0.0%):   2%|▏         | 1/60 [00:00<00:12,  4.78it/s]

2026/04/16 12:35:26 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=500. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.2)  if the reason for truncation is repetition.


...

Average Metric: 0.00 / 13 (0.0%): 100%|██████████| 14/14 [01:12<00:00,  5.16s/it]

2026/04/16 12:45:05 INFO dspy.evaluate.evaluate: Average Metric: 0.0 / 14 (0.0%)
2026/04/16 12:45:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 12:45:05 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/rob/llm-based-cluster-validation
2026/04/16 12:45:05 WARNING mlflow.dspy.save: Saving DSPy model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended alternative is to set 'use_dspy_model_save' to True (requiring dspy >= 3.1.0) to save the DSPy model using the DSPy builtin saving method.




Optimized dev accuracy : 1.7%  (delta: +1.7%)
Optimized test accuracy: 0.0%


2026/04/16 12:45:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.



Compiled program saved to /home/rob/llm-based-cluster-validation/outputs/program_optimized.json
Summary saved to /home/rob/llm-based-cluster-validation/outputs/optimize_results.json

BootstrapFewShot test accuracy: 0.0%
Delta vs baseline            : -1.1%


---

## Section 3: GEPA

### How GEPA works

GEPA (Generalized Efficient Prompt Adaptation) goes beyond few-shot selection and
**rewrites the task instructions** themselves. It operates in a reflection loop:

1. Run the current program on the training set, collect failures.
2. Feed failures to a *reflection LM*, which proposes revised instructions.
3. Score the new instructions on the validation set.
4. Repeat, keeping a Pareto frontier of instruction candidates.

GEPA needs a **70/15/15 train/val/dev split** because it uses train for reflection
updates and val for Pareto scoring — evaluation on dev is held out until the end.

The reflection LM should be stronger than the task LM. Here it is read from the
`teacher` block in `config/dspy_config.yaml` — currently `claude-sonnet-4-6` via
the Anthropic API. `ministral-3:3b` remains the task LM.

**Requires `ANTHROPIC_API_KEY` to be set.**

In [ ]:
from pipeline.optimize_gepa import run_optimization as run_gepa

# The reflection LM (claude-sonnet-4-6) is read from config/dspy_config.yaml.
# Requires ANTHROPIC_API_KEY to be set in your environment.
gepa_score = run_gepa(
    config_path=CONFIG_PATH,
    auto="light",   # "light" / "medium" / "heavy" — controls iteration budget
    num_threads=4,
)
print(f"\nGEPA test accuracy  : {gepa_score:.1f}%")
print(f"Delta vs baseline   : {gepa_score - baseline_score:+.1f}%")

---

## Section 4: BootstrapFinetune

### How BootstrapFinetune works

BootstrapFinetune is a **weight-level** optimizer — it actually updates model parameters.
It uses a teacher-student setup:

| Role    | Model              | What it does                                        |
|---------|--------------------|-----------------------------------------------------|
| Teacher | `claude-sonnet-4-6`| Runs inference on the trainset; provides gold traces|
| Student | `ministral-3:3b`   | Gets fine-tuned on those traces via TRL/PEFT        |

**Why this matters:**  
Prompt optimizers (Sections 2–3) only change *what we say* to the model. BootstrapFinetune
changes *what the model knows*, making the improvement permanent and inference-time-free —
the fine-tuned student no longer needs few-shot examples in the prompt.

**The two-phase process:**
1. The teacher (Claude Sonnet) runs over the trainset. Only examples where the teacher
   answers correctly become training data — this is the "bootstrap" part.
2. The collected (prompt, completion) pairs are passed to HuggingFace TRL for SFT
   (supervised fine-tuning) of the student model.

**Requirements:** ANTHROPIC_API_KEY must be set, SGLang must be running.

In [ ]:
import os

# Verify the API key is available before starting the long-running job
if not os.getenv("ANTHROPIC_API_KEY"):
    raise EnvironmentError(
        "ANTHROPIC_API_KEY is not set. "
        "Add it to your .env file or export it in the terminal before running this cell."
    )

print("ANTHROPIC_API_KEY is set — proceeding with BootstrapFinetune.")

In [ ]:
from pipeline.optimize_finetune import run_optimization as run_finetune

# This cell is long-running: it calls Claude Sonnet on the trainset,
# launches SGLang for the student, and runs SFT training.
finetune_score = run_finetune(
    config_path=CONFIG_PATH,
    output_dir="./finetuned_model",
    epochs=1,
    use_peft=True,      # LoRA — reduces GPU memory requirements
    num_threads=4,
)
print(f"\nBootstrapFinetune test accuracy: {finetune_score:.1f}%")
print(f"Delta vs baseline              : {finetune_score - baseline_score:+.1f}%")

---

## Results summary

In [ ]:
results = {
    "Baseline (no optimization)": baseline_score,
    "BootstrapFewShot": bootstrap_score,
    "GEPA": gepa_score,
    "BootstrapFinetune": finetune_score,
}

print(f"{'Method':<30} {'Test accuracy':>15} {'Delta':>10}")
print("-" * 57)
for method, score in results.items():
    delta = score - baseline_score
    delta_str = f"{delta:+.1f}%" if method != "Baseline (no optimization)" else "—"
    print(f"{method:<30} {score:>14.1f}% {delta_str:>10}")

print()
print("Full traces and metrics are available in the MLflow UI.")
print("Run: uv run mlflow ui   →   http://localhost:5000")